<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Launch Sites Locations Analysis with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:
- **TASK 1:** Mark all launch sites on a map
- **TASK 2:** Mark the success/failed launches for each site on the map
- **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [1]:
# !pip3 install folium
# !pip3 install wget
# !pip3 install pandas

In [2]:
import os
import folium
import wget
import pandas as pd

In [3]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/DV0101EN-3-5-1-Generating-Maps-in-Python-py-v2.0.ipynb)


## Task 1: Mark all launch sites on a map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site. 


In [4]:
# Download and read the `spacex_launch_geo.csv`
filepath = '../data/raw/spacex_launch_geo.csv'
if os.path.exists(filepath):
    print('File already exists, reading file...')
    spacex_df = pd.read_csv(filepath)
else:
    wget.download('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv', out='../data/raw')
    spacex_df = pd.read_csv(filepath)
    print('File downloaded and read...')

File already exists, reading file...


In [5]:
spacex_df.head()

,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356


Now, you can take a look at what are the coordinates for each site.


In [6]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df = launch_sites_df.rename(columns={'Launch Site': 'Launch_Site'})
launch_sites_df

,Launch_Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


- Basemap config

In [7]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
#Initializing Map object
site_map = folium.Map(
    location=nasa_coordinate,
    zoom_start=7, 
)
if not os.path.exists:
    site_map.save('../maps/basemap.html')

site_map

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example, 


In [8]:

# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
folium.Circle(
    location=nasa_coordinate,
    radius=1000,
    color='blue',
    fill=True, 
    popup='Nasa Johnson Space Center'
).add_to(site_map)

label_html = f"""
    <div style="
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;
        font-size: 11px;
        font-weight: 600;
        letter-spacing: 0.5px;
        color: 'blue';
        background: 'grey';
        border: 1px solid 'blue'44;
        border-radius: 4px;
        padding: 2px 7px;
        white-space: nowrap;
        box-shadow: 0 2px 6px rgba(0,0,0,0.6);
        pointer-events: none;
    ">
        NASA JSC
    </div>
    """

folium.Marker(
    location=nasa_coordinate,
    icon=DivIcon(
            icon_size=(150, 24),    # Explicit bounding box prevents truncation
            icon_anchor=(-10, 12),  # Offsets label 10px to the right and vertically centered
            html=label_html,
        )
).add_to(site_map)

if not os.path.exists('../maps/basewcircles.html'):
    site_map.save('../maps/basewcircles.html')

site_map

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle. 


Now, let's add a circle for each launch site in data frame `launch_sites`


_TODO:_  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [9]:
#Loop to access dataframe values
for row in launch_sites_df.itertuples():
    # Outer radius ring (1000m)
    folium.Circle(
        location=[row.Lat, row.Long],
        radius=1000,
        color='blue',
        weight=1.5,
        fill=True,
        fill_color='blue',
        fill_opacity=0.15,
        interactive=False,
        popup=folium.Popup(row.Launch_Site, parse_html=True)
    ).add_to(site_map)

    # Center focal point (small dot so the location is pinpointed)
    folium.CircleMarker(
        location=[row.Lat, row.Long],
        radius=4,
        color="#ffffff",
        fill=True,
        fill_color='blue',
        fill_opacity=1.0,
        weight=1.5
    ).add_to(site_map)

    # Styled Label Marker
    label_html = f"""
    <div style="
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;
        font-size: 11px;
        font-weight: 600;
        letter-spacing: 0.5px;
        color: 'blue;
        background: 'grey';
        border: 1px solid 'blue'44;
        border-radius: 4px;
        padding: 2px 7px;
        white-space: nowrap;
        box-shadow: 0 2px 6px rgba(0,0,0,0.6);
        pointer-events: none;
    ">
        {row.Launch_Site}
    </div>
    """

    folium.Marker(
        location=[row.Lat, row.Long],
        icon=DivIcon(
            icon_size=(150, 24),    # Explicit bounding box prevents truncation
            icon_anchor=(-10, 12),  # Offsets label 10px to the right and vertically centered
            html=label_html,
        )
    ).add_to(site_map)


if not os.path.exists('../maps/launchSites.html'):
    site_map.save('../maps/launchSites.html')

site_map

<!-- The generated map with marked launch sites should look similar to the following: -->


<!-- <center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center> -->


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:
- Are all launch sites in proximity to the Equator line?
- Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


# Task 2: Mark the success/failed launches for each site on the map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [10]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


Next, let's create markers for all launch records. 
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [11]:
marker_cluster = MarkerCluster()

_TODO:_ Create a new column in `launch_sites` dataframe called `marker_color` to store the marker colors based on the `class` value


In [12]:
spacex_df['marker_color'] = spacex_df['class'].apply(lambda x: 'red' if x == 0 else 'green')
spacex_df.head()

,Launch Site,Lat,Long,class,marker_color
0,CCAFS LC-40,28.562302,-80.577356,0,red
1,CCAFS LC-40,28.562302,-80.577356,0,red
2,CCAFS LC-40,28.562302,-80.577356,0,red
3,CCAFS LC-40,28.562302,-80.577356,0,red
4,CCAFS LC-40,28.562302,-80.577356,0,red


- deprecated

In [13]:
# # Function to assign color to launch outcome
# def assign_marker_color(launch_outcome):
#     if launch_outcome == 1:
#         return 'green'
#     else:
#         return 'red'
    
# spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
# spacex_df.tail(10)

_TODO:_ For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [14]:
spacex_df = spacex_df.rename(columns={'Launch Site': 'Launch_Site'})
spacex_df.head()

,Launch_Site,Lat,Long,class,marker_color
0,CCAFS LC-40,28.562302,-80.577356,0,red
1,CCAFS LC-40,28.562302,-80.577356,0,red
2,CCAFS LC-40,28.562302,-80.577356,0,red
3,CCAFS LC-40,28.562302,-80.577356,0,red
4,CCAFS LC-40,28.562302,-80.577356,0,red


In [15]:
site_map.add_child(marker_cluster) 

for row in spacex_df.itertuples():

    marker = folium.Marker(
        location=[row.Lat, row.Long],
        popup=folium.Popup(row.Launch_Site),
        icon=folium.Icon(
            color='white',
            icon_color=row.marker_color,
            icon='info-sign'
        )
    ).add_to(marker_cluster)



if not os.path.exists('../maps/launchresults.html'):
    site_map.save('../maps/launchresults.html')
    
site_map

- deprecated

In [16]:
# # Add marker_cluster to current site_map
# site_map.add_child(marker_cluster)

# # for each row in spacex_df data frame
# # create a Marker object with its coordinate
# # and customize the Marker's icon property to indicate if this launch was successed or failed, 
# # e.g., icon=folium.Icon(color='white', icon_color=row['marker_color']
# for index, record in spacex_df.iterrows():
#     # TODO: Create and add a Marker cluster to the site map
#     # marker = folium.Marker(...)
#     marker_cluster.add_child(marker)

# site_map

<!-- Your updated map may look like the following screenshots: -->


<!-- <center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center> -->


<!-- <center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center> -->


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


# TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [17]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
).add_to(site_map)

site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


You can calculate the distance between two points on the map based on their `Lat` and `Long` values using the following method:


In [18]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

_TODO:_ Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [19]:
# find coordinate of the closet coastline
coastline_lat, coastline_lon = [28.5633, -80.56789]
launch_site_lat, launch_site_lon = [28.56321, -80.57683]

distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

_TODO:_ After obtained its coordinate, create a `folium.Marker` to show the distance


In [20]:
# Create and add a folium.Marker on your selected closest coastline point on the map
# Display the distance between coastline point and launch site using the icon property 
# for example
distance_marker = folium.Marker(
   location=[coastline_lat, coastline_lon],
   icon=DivIcon(
       icon_size=(20,20),
       icon_anchor=(0,0),
       html=f'<div style="font-size: 12; color:#d35400;"><b>{distance_coastline:10.2f}KM</b></div>'
       )
   ).add_to(site_map)

_TODO:_ Draw a `PolyLine` between a launch site to the selected coastline point


In [21]:
# Create a `folium.PolyLine` object using the coastline coordinates and launch site coordinate
lines=folium.PolyLine(locations=[(coastline_lat, coastline_lon),(launch_site_lat, launch_site_lon)], weight=1)
site_map.add_child(lines)
site_map

Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


_TODO:_ Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [22]:
from folium import ClickForMarker
ClickForMarker(popup='pin').add_to(site_map)
site_map

In [23]:
#Create a marker with distance to a closest city, railway, highway, etc.
# Draw a line between the marker to the launch site
rail_coordinates = [28.56345, -80.58691]
highway_coordinates = [28.56338, -80.58696]
city_coordinates = [28.0744, -80.62592]



folium.Marker(
   location=[rail_coordinates[0], rail_coordinates[1]],
   icon=DivIcon(
       icon_size=(20,20),
       icon_anchor=(0,0),
       html=f'<div style="font-size: 12; color:#d35400;"><b>{calculate_distance(rail_coordinates[0], rail_coordinates[1], launch_site_lat, launch_site_lon):10.2f}KM</b></div>'
       )
   ).add_to(site_map)

folium.PolyLine(locations=[(rail_coordinates[0], rail_coordinates[1]),(launch_site_lat, launch_site_lon)], weight=2).add_to(site_map)


folium.Marker(
   location=[highway_coordinates[0], highway_coordinates[1]],
   icon=DivIcon(
       icon_size=(20,20),
       icon_anchor=(0,0),
       html=f'<div style="font-size: 12; color:#d35400;"><b>{calculate_distance(highway_coordinates[0], highway_coordinates[1], launch_site_lat, launch_site_lon):10.2f}KM</b></div>'
       )
   ).add_to(site_map)

folium.PolyLine(locations=[(highway_coordinates[0], highway_coordinates[1]),(launch_site_lat, launch_site_lon)], weight=2).add_to(site_map)


folium.Marker(
   location=[city_coordinates[0], city_coordinates[1]],
   icon=DivIcon(
       icon_size=(20,20),
       icon_anchor=(0,0),
       html=f'<div style="font-size: 12; color:#d35400;"><b>{calculate_distance(city_coordinates[0], city_coordinates[1], launch_site_lat, launch_site_lon):10.2f}KM</b></div>'
       )
   ).add_to(site_map)

folium.PolyLine(locations=[(city_coordinates[0], city_coordinates[1]),(launch_site_lat, launch_site_lon)], weight=2).add_to(site_map)

site_map

After you plot distance lines to the proximities, you can answer the following questions easily:
- Are launch sites in close proximity to railways?
- Are launch sites in close proximity to highways?
- Are launch sites in close proximity to coastline?
- Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


### Findings
- Railway -> 1KM,
- Highway -> 1KM,
- Coastline -> 0.8KM, 
- City -> 55KM

- Close proximity to transportation
- Relatively less proximity to large populations

# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Yan Luo](https://www.linkedin.com/in/yan-luo-96288783/)


### Other Contributors


Joseph Santarcangelo


## Change Log


|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2021-05-26|1.0|Yan|Created the initial version|


Copyright © 2021 IBM Corporation. All rights reserved.
